# Step 9A — Entropy Calculation
## Berbasis Kolom `tweet_tokens_stemmed`

Notebook ini menghitung **entropy** murni dari distribusi token hasil stemming.
Kolom `Relevansi` **tidak digunakan**.

### Entropy yang dihitung:
| # | Jenis Entropy | Basis |
|---|---|---|
| 1 | **Entropy per Dokumen** | Distribusi frekuensi token dalam setiap tweet |
| 2 | **Entropy Corpus Global** | Distribusi frekuensi semua token di seluruh dataset |
| 3 | **Kontribusi Entropy per Token** | Seberapa besar setiap kata unik berkontribusi ke entropy corpus |

## 1. Install & Import Library

In [ ]:
%pip install pandas numpy matplotlib seaborn -q

In [ ]:
import pandas as pd
import numpy as np
import ast
import re
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded!')

## 2. Upload & Load Data

Upload file `Step_8-Clean_Relevansi_Data.csv`.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload Step_8-Clean_Relevansi_Data.csv

In [ ]:
# Separator titik koma sesuai format file
df = pd.read_csv('Step_8-Clean_Relevansi_Data.csv', sep=';', on_bad_lines='skip')

print(f'Total baris          : {len(df)}')
print(f'Kolom tersedia       : {df.columns.tolist()}')
print()
print('Sample tweet_tokens_stemmed:')
for i in range(3):
    print(f'  [{i}] {str(df["tweet_tokens_stemmed"].iloc[i])[:120]}')

## 3. Parse Kolom `tweet_tokens_stemmed`

Kolom tersimpan sebagai string representasi list Python, perlu di-parse ke list.

In [ ]:
def parse_tokens(text):
    """
    Parse string list token menjadi list Python.
    Mendukung: ['kata1', 'kata2'] dan ['kata1' 'kata2']
    """
    if pd.isna(text) or str(text).strip() == '':
        return []
    text = str(text).strip()
    # Coba parse langsung
    try:
        result = ast.literal_eval(text)
        if isinstance(result, list):
            return [str(t).strip() for t in result if str(t).strip()]
    except:
        pass
    # Fallback: ekstrak token dari tanda kutip tunggal
    tokens = re.findall(r"'([^']+)'", text)
    return [t.strip() for t in tokens if t.strip()]


df['tokens'] = df['tweet_tokens_stemmed'].apply(parse_tokens)

valid     = df['tokens'].apply(len) > 0
df_valid  = df[valid].copy().reset_index(drop=True)

print(f'Dokumen valid (ada token) : {len(df_valid)}')
print(f'Dokumen kosong / NaN      : {(~valid).sum()}')
print()
print('Contoh hasil parsing:')
for i in range(3):
    print(f'  [{i}] {df_valid["tokens"].iloc[i]}')

## 4. Entropy per Dokumen

$$H(d_i) = -\sum_{t \in d_i} p(t|d_i) \cdot \log_2 p(t|d_i)$$

- **Entropy = 0** → semua token sama
- **Entropy tinggi** → token sangat beragam, tidak ada yang mendominasi
- **Entropy maksimal** = $\log_2(|\text{token unik}|)$, tercapai jika semua token berbeda

In [ ]:
def hitung_entropy(tokens):
    if not tokens:
        return 0.0
    counts = np.array(list(Counter(tokens).values()), dtype=float)
    probs  = counts / counts.sum()
    return float(-np.sum(probs * np.log2(probs + 1e-12)))


df_valid['n_token']       = df_valid['tokens'].apply(len)
df_valid['n_token_unik']  = df_valid['tokens'].apply(lambda x: len(set(x)))
df_valid['entropy_bit']   = df_valid['tokens'].apply(hitung_entropy)
df_valid['entropy_max']   = df_valid['n_token_unik'].apply(
    lambda n: np.log2(n) if n > 1 else 0.0
)
df_valid['entropy_norm']  = df_valid.apply(
    lambda r: r['entropy_bit'] / r['entropy_max'] if r['entropy_max'] > 0 else 0.0,
    axis=1
).round(6)

print('=' * 56)
print('  ENTROPY PER DOKUMEN (tweet_tokens_stemmed)')
print('=' * 56)
print(f'  Jumlah dokumen          : {len(df_valid)}')
print()
for label, col in [
    ('Entropy (bit) — Mean  ', 'entropy_bit'),
    ('Entropy (bit) — Median', 'entropy_bit'),
    ('Entropy (bit) — Std   ', 'entropy_bit'),
    ('Entropy (bit) — Min   ', 'entropy_bit'),
    ('Entropy (bit) — Max   ', 'entropy_bit'),
]:
    fn = {'Mean  ': 'mean', 'Median': 'median', 'Std   ': 'std',
          'Min   ': 'min',  'Max   ': 'max'}[label.split('— ')[1]]
    val = getattr(df_valid[col], fn)()
    print(f'  {label}: {val:.6f} bit')

print()
print(f'  Entropy Ternormalisasi — Mean  : {df_valid["entropy_norm"].mean():.6f}')
print(f'  Rata-rata token/dokumen        : {df_valid["n_token"].mean():.2f}')
print(f'  Rata-rata token unik/dokumen   : {df_valid["n_token_unik"].mean():.2f}')

In [ ]:
print('=== 5 Tweet dengan Entropy TERTINGGI ===')
for _, r in df_valid.nlargest(5, 'entropy_bit').iterrows():
    print(f"  No={int(r['No'])}  n_tok={int(r['n_token'])}  unik={int(r['n_token_unik'])}  H={r['entropy_bit']:.6f} bit")
    print(f"  >> {str(r['full_text'])[:100]}")
    print()

print('=== 5 Tweet dengan Entropy TERENDAH (>0) ===')
for _, r in df_valid[df_valid['entropy_bit'] > 0].nsmallest(5, 'entropy_bit').iterrows():
    print(f"  No={int(r['No'])}  n_tok={int(r['n_token'])}  unik={int(r['n_token_unik'])}  H={r['entropy_bit']:.6f} bit")
    print(f"  >> {str(r['full_text'])[:100]}")
    print()

## 5. Visualisasi Entropy per Dokumen

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

mean_h = df_valid['entropy_bit'].mean()
med_h  = df_valid['entropy_bit'].median()

# 1. Histogram entropy
axes[0].hist(df_valid['entropy_bit'], bins=40,
             color='#2196F3', edgecolor='white', alpha=0.85)
axes[0].axvline(mean_h, color='red',    ls='--', lw=2, label=f'Mean {mean_h:.3f}')
axes[0].axvline(med_h,  color='orange', ls='--', lw=2, label=f'Median {med_h:.3f}')
axes[0].set_xlabel('Entropy (bit)', fontsize=11)
axes[0].set_ylabel('Jumlah Dokumen', fontsize=11)
axes[0].set_title('Distribusi Entropy\nper Dokumen', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)

# 2. Scatter: n_token vs entropy
axes[1].scatter(df_valid['n_token'], df_valid['entropy_bit'],
                alpha=0.4, color='#673AB7', s=20, edgecolors='none')
axes[1].set_xlabel('Jumlah Token', fontsize=11)
axes[1].set_ylabel('Entropy (bit)', fontsize=11)
axes[1].set_title('Jumlah Token vs Entropy', fontsize=12, fontweight='bold')

# 3. Scatter: n_token_unik vs entropy
axes[2].scatter(df_valid['n_token_unik'], df_valid['entropy_bit'],
                alpha=0.4, color='#009688', s=20, edgecolors='none')
axes[2].set_xlabel('Token Unik', fontsize=11)
axes[2].set_ylabel('Entropy (bit)', fontsize=11)
axes[2].set_title('Token Unik vs Entropy', fontsize=12, fontweight='bold')

plt.suptitle('Entropy per Dokumen — tweet_tokens_stemmed',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('entropy_per_dokumen.png', dpi=150, bbox_inches='tight')
plt.show()
print('Tersimpan: entropy_per_dokumen.png')

## 6. Entropy Corpus Global

$$H_{corpus} = -\sum_{w \in V} p(w) \cdot \log_2 p(w)$$

- $V$ = vocabulary seluruh corpus
- $p(w)$ = frekuensi relatif kata $w$

In [ ]:
all_tokens = []
for tok in df_valid['tokens']:
    all_tokens.extend(tok)

token_freq  = Counter(all_tokens)
total_tok   = len(all_tokens)
vocab_size  = len(token_freq)

freqs_arr      = np.array(list(token_freq.values()), dtype=float)
probs_corpus   = freqs_arr / total_tok
entropy_corpus = float(-np.sum(probs_corpus * np.log2(probs_corpus + 1e-12)))
entropy_max    = np.log2(vocab_size)

print('=' * 52)
print('  ENTROPY CORPUS (tweet_tokens_stemmed)')
print('=' * 52)
print(f'  Total token corpus           : {total_tok:,}')
print(f'  Ukuran vocabulary (unik)     : {vocab_size:,}')
print()
print(f'  Entropy Corpus               : {entropy_corpus:.6f} bit')
print(f'  Entropy Maksimal (uniform)   : {entropy_max:.6f} bit')
print(f'  Utilisasi Entropy            : {entropy_corpus / entropy_max * 100:.2f}%')
print()

# Tabel Top 20
print(f'{"Rank":>4}  {"Token":<22} {"Freq":>6}  {"p(w)":>9}  {"H_kontrib":>10}')
print('-' * 58)
for rank, (tok, freq) in enumerate(token_freq.most_common(20), 1):
    p  = freq / total_tok
    hc = -p * np.log2(p + 1e-12)
    print(f'{rank:>4}  {tok:<22} {freq:>6}  {p:>9.5f}  {hc:>10.6f}')

## 7. Visualisasi Corpus

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar top 20
top20 = pd.DataFrame(token_freq.most_common(20), columns=['token', 'frekuensi'])
sns.barplot(data=top20, x='frekuensi', y='token', palette='Blues_d', ax=axes[0])
axes[0].set_xlabel('Frekuensi', fontsize=11)
axes[0].set_ylabel('Token', fontsize=11)
axes[0].set_title('Top 20 Token Paling Sering\n(tweet_tokens_stemmed)',
                  fontsize=12, fontweight='bold')

# Histogram frekuensi (log scale) — Zipf's Law
axes[1].hist(list(token_freq.values()), bins=60,
             color='#FF9800', edgecolor='white', alpha=0.85)
axes[1].set_yscale('log')
axes[1].set_xlabel('Frekuensi Token', fontsize=11)
axes[1].set_ylabel('Jumlah Token Unik (log)', fontsize=11)
axes[1].set_title(
    f'Distribusi Frekuensi Vocabulary\n'
    f'Entropy = {entropy_corpus:.4f} bit  |  Vocab = {vocab_size:,}',
    fontsize=12, fontweight='bold'
)

plt.tight_layout()
plt.savefig('entropy_corpus.png', dpi=150, bbox_inches='tight')
plt.show()
print('Tersimpan: entropy_corpus.png')

## 8. Simpan Hasil

In [ ]:
# CSV 1: entropy per dokumen
out_dok = df_valid[['No', 'full_text', 'username', 'tweet_tokens_stemmed',
                     'n_token', 'n_token_unik', 'entropy_bit',
                     'entropy_max', 'entropy_norm']].copy()
out_dok.to_csv('Step_9A_Entropy_per_Dokumen.csv', index=False)

# CSV 2: kontribusi entropy per token vocabulary
vocab_df = pd.DataFrame([
    {
        'token'        : t,
        'frekuensi'    : f,
        'prob'         : round(f / total_tok, 8),
        'H_kontribusi' : round(-(f/total_tok) * np.log2(f/total_tok + 1e-12), 8)
    }
    for t, f in token_freq.most_common()
])
vocab_df.to_csv('Step_9A_Vocab_Entropy.csv', index=False)

print('File tersimpan:')
print('  Step_9A_Entropy_per_Dokumen.csv  — entropy tiap tweet')
print('  Step_9A_Vocab_Entropy.csv        — frekuensi & H kontribusi per token')
print()
print('=' * 52)
print('  RINGKASAN AKHIR')
print('=' * 52)
print(f'  Dokumen diproses     : {len(df_valid)}')
print(f'  Total token          : {total_tok:,}')
print(f'  Vocabulary           : {vocab_size:,} kata unik')
print(f'  Entropy Corpus       : {entropy_corpus:.6f} bit')
print(f'  Utilisasi Entropy    : {entropy_corpus / entropy_max * 100:.2f}%')
print(f'  H/dok mean           : {df_valid["entropy_bit"].mean():.6f} bit')
print(f'  H/dok median         : {df_valid["entropy_bit"].median():.6f} bit')

out_dok.head()

In [ ]:
from google.colab import files
files.download('Step_9A_Entropy_per_Dokumen.csv')
files.download('Step_9A_Vocab_Entropy.csv')
files.download('entropy_per_dokumen.png')
files.download('entropy_corpus.png')